In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
import feyn

# Model agnostic 
from typing import Optional, List, Callable, Dict, Any, List
from pathlib import Path
from src.utils import Helm, QLatticeWrapper

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

In [2]:
# Get the directory this file lives in
nb_dir = Path.cwd() # notebook directory
project_root = nb_dir.parents[0] # project directory
data_path = project_root / "datasets" / "processed_well_data.csv"

includ_cols = ['Dia', 'Dev(deg)','Area (m2)', 'z','GasDens','LiquidDens', 'P/T','friction_factor', 'critical_film_thickness']
D = Helm(path=data_path, includ_cols=includ_cols, test_size=0.20)

In [3]:
# define xgboost pipeline
def qlattice(
        hparams: Dict[str,Any]
):
    ql_wrap = QLatticeWrapper(
        feature_tags=includ_cols, 
        **hparams,
    )

    return ql_wrap

hparam_grid = {
            "max_complexity":  [25], # [15, 25, 35],
            "n_epochs":  [15], #[10, 15, 20],
        }

hparam_grid["interval"] = [0]  # Classification tolerance for well status

# train model and optimize hyperparameters via grid search 
trained_model = D.evolv_model(build_model=qlattice, hparam_grid=hparam_grid, k_folds=5)

# output equation 
print(trained_model.express())


Best CV Classification Accuracy: 
>>> 0.6383 ± 0.0617

Best CV Per-Class Accuracy:
>>> Unloaded (-1): 0.3977, Near L.U. (0): 0.0000, Loaded (1): 0.9726

Best Hyperparameters:
>>> {'max_complexity': 25, 'n_epochs': 15, 'interval': 0}

Training Classification Accuracy:
>>> 0.6627

Per-Class Training Accuracy:
>>> Unloaded (-1): 0.4205, Near L.U. (0): 0.0000, Loaded (1): 1.0000

Training Regression Metrics:
>>> RMSE=82175.2567, MAE=64973.2452, R2=0.5730

Test Regression Metrics:
>>> RMSE=85817.5785, MAE=71625.1722, R2=0.5033

Test Classification Accuracy:
>>> 0.6429

Per-Class Test Accuracy:
>>> Unloaded (-1): 0.3913, Near L.U. (0): 0.0000, Loaded (1): 1.0000


0.327157*P/T + 1.65672*tanh(0.860137*Area (m2) + 0.271018*criticalfilmthickness + 0.388907) - 0.250776
